# 10-3절 연습 문제 풀이

디코더만 사용하는 트랜스포머(`OzWriterTransformer`)와 세 가지 생성 전략을 다루는
[연습 문제 10-12] ~ [연습 문제 10-16]의 풀이다.

본문 예제(`10-03_example.ipynb`)의 데이터, 모델, 학습 함수를 그대로 사용한다.


## 공통 준비

In [1]:

import sys
sys.path.append('../../')

from code_reference import common
from code_reference import visualize as viz

viz.configure(save_grayscale=False)
common.set_korean_plot_env()

SEED = 42
common.set_seed(SEED)
device = common.get_device()

CUDA를 사용합니다.


In [2]:

# 오즈의 마법사 텍스트 로딩과 토큰화(본문 예제와 동일)
from pathlib import Path
import re

raw = Path('../../data/wonderful_wizard_of_oz.txt').read_text(encoding='utf-8-sig')
match = re.search(r'\nChapter I\n', raw)
start = match.start() if match else raw.find('Chapter I')
end = raw.rfind('*** END OF THE PROJECT GUTENBERG')
if end == -1:
    end = raw.rfind('THE END')
text = raw[start:end].strip()

normalized = text.lower()
normalized = re.sub(r"[^a-z\s,.!?\u2019']", ' ', normalized)
tokens = re.findall(r"[a-z\u2019']+|[,.!?]", normalized)
tokens = [t.rstrip('\u2019') if not t.endswith("'") else t for t in tokens]
tokens = [t for t in tokens if t]

print(f'전체 토큰 수 {len(tokens):,}, 고유 토큰 수 {len(set(tokens)):,}')

전체 토큰 수 44,299, 고유 토큰 수 2,902


In [3]:

# 어휘 사전, 데이터셋, 데이터로더([코드 10-12])
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

vocab = {t: i for i, t in enumerate(sorted(set(tokens)))}
vocab_size = len(vocab)
itos = {i: t for t, i in vocab.items()}


class OzTransformerDataset(Dataset):
    """전체 토큰을 한 토큰씩 겹쳐 seq_length + 1 길이로 분할(비중첩 분할)"""

    def __init__(self, token_idxs, seq_length):
        self.token_idxs = token_idxs
        self.seq_length = seq_length

    def __len__(self):
        return (len(self.token_idxs) - 1) // self.seq_length

    def __getitem__(self, idx):
        start = idx * self.seq_length
        end = start + self.seq_length + 1
        return torch.tensor(self.token_idxs[start:end], dtype=torch.long)


SEQ_LENGTH = 32
BATCH_SIZE = 64
token_idxs = [vocab[t] for t in tokens]
split = int(len(token_idxs) * 0.8)

train_set = OzTransformerDataset(token_idxs[:split], SEQ_LENGTH)
valid_set = OzTransformerDataset(token_idxs[split:], SEQ_LENGTH)
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_set, batch_size=BATCH_SIZE, shuffle=False)

print(f'어휘 사전 {vocab_size}, 훈련/검증 샘플 {len(train_set)} / {len(valid_set)}')

어휘 사전 2902, 훈련/검증 샘플 1107 / 276


In [4]:

# 위치 인코딩과 모델([코드 10-1], [코드 10-11])
class PositionalEncoding(nn.Module):
    def __init__(self, max_length, d_model):
        super().__init__()
        self.position_embedding = nn.Embedding(max_length, d_model)
        self.activation = nn.Tanh()

    def forward(self, token_embedded):
        seq_length = token_embedded.size(1)
        positions = torch.arange(seq_length, device=token_embedded.device)
        return self.activation(token_embedded + self.position_embedding(positions))


class OzWriterTransformer(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, ff_dim,
                 num_layers, max_length, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(max_length, d_model)
        self.dropout = nn.Dropout(dropout)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=num_heads, dim_feedforward=ff_dim,
            dropout=dropout, batch_first=True
        )
        self.decoder = nn.TransformerEncoder(
            encoder_layer, num_layers=num_layers, enable_nested_tensor=False
        )
        self.fc = nn.Linear(d_model, vocab_size)

    def forward(self, src):
        seq_length = src.size(1)
        causal_mask = nn.Transformer.generate_square_subsequent_mask(
            seq_length, device=src.device
        )
        embedded = self.dropout(self.pos_encoding(self.embedding(src)))
        output = self.decoder(embedded, mask=causal_mask, is_causal=True)
        return self.fc(output)

In [5]:

# 학습 함수(본문 예제와 동일, 조기 종료 없음)
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    loss_sum, sample_size = 0.0, 0
    for chunk in loader:
        chunk = chunk.to(device)
        src, tgt = chunk[:, :-1], chunk[:, 1:]
        optimizer.zero_grad()
        logits = model(src)
        loss = criterion(logits.reshape(-1, logits.size(-1)), tgt.reshape(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        batch_size = src.size(0)
        loss_sum += loss.item() * batch_size
        sample_size += batch_size
    return loss_sum / sample_size


@torch.no_grad()
def validation(model, loader, criterion, device):
    model.eval()
    loss_sum, sample_size = 0.0, 0
    for chunk in loader:
        chunk = chunk.to(device)
        src, tgt = chunk[:, :-1], chunk[:, 1:]
        logits = model(src)
        loss = criterion(logits.reshape(-1, logits.size(-1)), tgt.reshape(-1))
        batch_size = src.size(0)
        loss_sum += loss.item() * batch_size
        sample_size += batch_size
    return loss_sum / sample_size


def train_loop(model, train_loader, valid_loader, criterion, optimizer,
               epochs, device, name='', verbose_rows=10):
    model.to(device)
    log = common.EpochLogger(
        epochs, columns=('훈련 손실', '검증 손실'),
        formats=('{:.4f}', '{:.4f}'), target_rows=verbose_rows,
    )
    if name:
        print(f'{name} 학습')
    for epoch in range(1, epochs + 1):
        train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
        valid_loss = validation(model, valid_loader, criterion, device)
        log.row(epoch, train_loss, valid_loss)
    log.summary()
    return log


D_MODEL, NUM_HEADS, FF_DIM = 128, 4, 256
NUM_LAYERS, MAX_LENGTH, DROPOUT = 2, 64, 0.1
LR, EPOCHS = 1e-3, 100


def build_model(vsize=None, max_length=MAX_LENGTH):
    return OzWriterTransformer(
        vocab_size=vsize or vocab_size, d_model=D_MODEL, num_heads=NUM_HEADS,
        ff_dim=FF_DIM, num_layers=NUM_LAYERS, max_length=max_length, dropout=DROPOUT,
    ).to(device)

In [6]:

# 본문 예제와 같은 기준 모델 학습(이후 문제에서 공통으로 사용)
common.set_seed(SEED)
model = build_model()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

log_base = train_loop(model, train_loader, valid_loader, criterion, optimizer,
                      epochs=EPOCHS, device=device, name='기준 모델(비중첩 분할)')

기준 모델(비중첩 분할) 학습


 에포크    훈련 손실    검증 손실     시간
  1/100       7.0416       6.0981     0:01


 10/100       4.6218       5.2171     0:02


 20/100       3.5990       5.2102     0:03


 30/100       2.8655       5.5403     0:05


...

 60/100       1.7036       6.7620     0:09


 70/100       1.4887       7.1292     0:10


 80/100       1.3434       7.4379     0:11


 90/100       1.2192       7.7035     0:13


100/100       1.1104       7.9784     0:14
------------------------------------------
최적 14 에포크 · 검증 손실 5.1317 · 전체 학습 시간 0:14


In [7]:

# 본문의 세 생성 함수([코드 10-13], [코드 10-14], [코드 10-15])
@torch.no_grad()
def generate_greedy(model, prompt, vocab, seq_length, max_new_words=10, device='cpu'):
    model.eval()
    input_ids = [vocab[w] for w in prompt.lower().split()]
    for _ in range(max_new_words):
        src = torch.tensor(input_ids[-seq_length:], dtype=torch.long)
        src = src.unsqueeze(0).to(device)
        logits = model(src)
        input_ids.append(logits[0, -1, :].argmax().item())
    return ' '.join(itos[i] for i in input_ids)


@torch.no_grad()
def generate_temperature(model, prompt, vocab, seq_length, max_new_words=10,
                         temperature=1.0, device='cpu'):
    model.eval()
    input_ids = [vocab[w] for w in prompt.lower().split()]
    for _ in range(max_new_words):
        src = torch.tensor(input_ids[-seq_length:], dtype=torch.long)
        src = src.unsqueeze(0).to(device)
        logits = model(src)
        probs = F.softmax(logits[0, -1, :] / temperature, dim=-1)
        input_ids.append(torch.multinomial(probs, num_samples=1).item())
    return ' '.join(itos[i] for i in input_ids)


@torch.no_grad()
def generate_beam_search(model, prompt, vocab, seq_length, max_new_words=10,
                         beam_width=3, device='cpu'):
    model.eval()
    init_ids = [vocab[w] for w in prompt.lower().split()]
    beams = [(0.0, init_ids)]
    for _ in range(max_new_words):
        candidates = []
        for log_prob, ids in beams:
            src = torch.tensor(ids[-seq_length:], dtype=torch.long)
            src = src.unsqueeze(0).to(device)
            log_probs = F.log_softmax(model(src)[0, -1, :], dim=-1)
            topk_log_probs, topk_ids = log_probs.topk(beam_width)
            for lp, nid in zip(topk_log_probs.tolist(), topk_ids.tolist()):
                candidates.append((log_prob + lp, ids + [nid]))
        beams = sorted(candidates, key=lambda x: x[0], reverse=True)[:beam_width]
    return ' '.join(itos[i] for i in beams[0][1])


PROMPT_A = 'dorothy looked at'
PROMPT_B = 'in the middle'
print(generate_greedy(model, PROMPT_A, vocab, SEQ_LENGTH, device=device))

dorothy looked at her companions , and put them back to the little


## 연습 문제 10-12

In [8]:

# Top-k 샘플링을 결합한 생성 함수
@torch.no_grad()
def generate_topk(model, prompt, vocab, seq_length, max_new_words=10,
                  temperature=1.0, k=10, device='cpu'):
    """온도 샘플링에서 상위 k개 토큰만 남기고 확률을 다시 계산해 샘플링한다."""
    model.eval()
    input_ids = [vocab[w] for w in prompt.lower().split()]
    for _ in range(max_new_words):
        src = torch.tensor(input_ids[-seq_length:], dtype=torch.long)
        src = src.unsqueeze(0).to(device)
        scaled_logits = model(src)[0, -1, :] / temperature
        # 상위 k개만 남기고 나머지는 -inf로 만들어 소프트맥스 확률을 0으로 보냄
        topk_logits, topk_ids = scaled_logits.topk(k)
        filtered = torch.full_like(scaled_logits, float('-inf'))
        filtered[topk_ids] = topk_logits
        probs = F.softmax(filtered, dim=-1)
        input_ids.append(torch.multinomial(probs, num_samples=1).item())
    return ' '.join(itos[i] for i in input_ids)


# k 값에 따른 결과 비교(온도 1.0 고정, k마다 4회 생성)
common.set_seed(SEED)
for k in (5, 10, 20):
    print(f'=== Top-{k} (온도 1.0) ===')
    for _ in range(4):
        print('  ' + generate_topk(model, PROMPT_B, vocab, SEQ_LENGTH,
                                   temperature=1.0, k=k, device=device))
    print()

=== Top-5 (온도 1.0) ===


  in the middle of the water , looking very lonely and sad .
  in the middle of the water , looking very lonely and sad .
  in the middle of the water , looking very lonely and sad .
  in the middle of the water , looking very lonely and sad .

=== Top-10 (온도 1.0) ===
  in the middle of the water , looking sober it , and sometimes
  in the middle of the water , and dorothy was near him up
  in the middle of the water , looking very lonely and sad .
  in the middle of the water , looking very lonely and sad .

=== Top-20 (온도 1.0) ===
  in the middle of the water , looking very lonely and sad .
  in the middle of the water , looking very lonely and sad .
  in the middle of the water , looking very lonely and sad .
  in the middle of the water , looking very lonely and sad .



In [9]:

# k가 실제로 확률 질량을 얼마나 잘라 내는지 수치로 확인
@torch.no_grad()
def topk_mass(model, prompt, k_list=(5, 10, 20, 50), temperature=1.0):
    model.eval()
    ids = [vocab[w] for w in prompt.lower().split()]
    src = torch.tensor(ids, dtype=torch.long).unsqueeze(0).to(device)
    probs = F.softmax(model(src)[0, -1, :] / temperature, dim=-1)
    sorted_probs, _ = probs.sort(descending=True)
    return {k: sorted_probs[:k].sum().item() for k in k_list}


print(f'프롬프트: {PROMPT_B!r} (어휘 사전 크기 {vocab_size})')
print('상위 k개 토큰이 차지하는 확률의 합')
for k, mass in topk_mass(model, PROMPT_B).items():
    print(f'  k={k:<3d} {mass * 100:5.1f}%')

프롬프트: 'in the middle' (어휘 사전 크기 2902)
상위 k개 토큰이 차지하는 확률의 합
  k=5   100.0%
  k=10  100.0%
  k=20  100.0%
  k=50  100.0%


### 풀이 해설 — 연습 문제 10-12

> **★ 이 문제와 다음 문제(10-12, 10-13)는 본문 모델에서 효과가 잘 드러나지 않는다.**
>
> 상위 5개 토큰이 이미 확률의 **100%**를 차지할 만큼 분포가 뾰족하기 때문이다(어휘
> 사전은 2,902개다). 본문 p32는 검증 손실이 **14 에포크에서 최저**라고 밝히면서도
> 전체 학습 에포크를 100으로 고정하는데, 그 결과 모델이 훈련 데이터를 상당히 외운
> 상태가 된다.
>
> 온도를 올려도, 상위 k개만 남겨도 **뽑히는 토큰이 달라지지 않는다.** 생성 전략이
> 작동할 여지 자체가 없다. 차이를 보고 싶다면 **학습 에포크를 20 안팎으로 줄인
> 모델**로 같은 실험을 해 보자.

---

**구현의 핵심은 '자른 뒤에 다시 정규화한다'는 것이다.** 상위 `k`개를 고른 다음 나머지
로짓을 `-inf`로 채우면, 소프트맥스가 그 자리를 0으로 만들면서 **남은 k개의 확률만으로
합이 1이 되도록 자동으로 다시 계산한다.** 잘라 낸 확률 질량을 손으로 나눠 줄 필요가 없다.

```python
topk_logits, topk_ids = scaled_logits.topk(k)
filtered = torch.full_like(scaled_logits, float('-inf'))
filtered[topk_ids] = topk_logits
probs = F.softmax(filtered, dim=-1)
```

순서도 중요하다. **온도로 먼저 나눈 뒤 잘라야 한다.** 반대로 하면 온도가 상위 k개
안에서의 분포만 바꾸므로, 온도와 k가 서로 다른 일을 하게 된다.

**확률 질량 표가 이 실험의 결론이다.**

| k | 확률 질량 |
|---|---|
| 5 | 100.0% |
| 10 | 100.0% |
| 20 | 100.0% |
| 50 | 100.0% |

**상위 5개가 이미 전부다.** 그래서 `k`를 5로 하든 20으로 하든 결과가 거의 같고, 네 번
생성해도 같은 문장이 반복된다.

**Top-k가 본래 막으려는 것**은 상위권의 경쟁이 아니라 **꼬리에 깔린 수천 개의 토큰이
우연히 뽑히는 사고**다. 이 모델에는 그 꼬리가 사실상 없으므로 막을 것도 없다.

**덜 학습된 모델이라면 이야기가 다르다.** 분포가 평평할수록 꼬리가 두꺼워지고, 그때
Top-k가 일을 한다. 확률 질량 표를 학습 에포크별로 재 보면 이 관계가 드러난다.

**Top-k의 구조적 약점도 짚어 둘 만하다.** `k`가 고정값이라 분포의 모양을 고려하지
못한다. 한 토큰이 확률 0.9를 차지하는 자리에서도 k개를 남기고, 상위 50개가 고만고만한
자리에서도 k개만 남긴다. 이 문제를 푸는 것이 학습 노트에 언급된 **Top-p(Nucleus)
샘플링**으로, 개수 대신 누적 확률을 기준으로 자른다.


## 연습 문제 10-13

In [10]:

# 빔 서치와 온도 샘플링을 결합한 생성 함수
@torch.no_grad()
def generate(model, prompt, vocab, seq_length, max_new_words=10,
             beam_width=3, temperature=1.0, k=0, penalty=0.0, window=8,
             device='cpu'):
    """빔 서치의 경로 탐색에 온도 샘플링의 확률적 선택을 결합한다.

    각 경로에서 상위 후보를 결정적으로 고르는 대신, 온도로 조절한 확률 분포에서
    beam_width개를 뽑는다. 경로 점수는 온도를 적용하지 않은 원래 로그 확률로
    누적해 경로끼리 공정하게 비교한다.
    """
    model.eval()
    init_ids = [vocab[w] for w in prompt.lower().split()]
    beams = [(0.0, init_ids)]
    for _ in range(max_new_words):
        candidates = []
        for log_prob, ids in beams:
            src = torch.tensor(ids[-seq_length:], dtype=torch.long)
            src = src.unsqueeze(0).to(device)
            logits = model(src)[0, -1, :].clone()
            if penalty > 0:
                for recent in set(ids[-window:]):
                    logits[recent] -= penalty
            # 경로 점수에 사용할 원래 로그 확률
            base_log_probs = F.log_softmax(logits, dim=-1)
            # 후보를 뽑을 때 사용할 온도 적용 분포
            sample_logits = logits / temperature
            if k > 0:
                topk_logits, topk_ids = sample_logits.topk(k)
                filtered = torch.full_like(sample_logits, float('-inf'))
                filtered[topk_ids] = topk_logits
                sample_logits = filtered
            probs = F.softmax(sample_logits, dim=-1)
            # 중복 없이 beam_width개 샘플링
            picked = torch.multinomial(probs, num_samples=beam_width,
                                       replacement=False).tolist()
            for nid in picked:
                candidates.append((log_prob + base_log_probs[nid].item(), ids + [nid]))
        beams = sorted(candidates, key=lambda x: x[0], reverse=True)[:beam_width]
    return ' '.join(itos[i] for i in beams[0][1])


common.set_seed(SEED)
print('빔 너비와 온도 조합 실험(프롬프트:', repr(PROMPT_A), ')\n')
for width in (3, 5):
    for temp in (0.7, 1.0, 1.3):
        outs = [generate(model, PROMPT_A, vocab, SEQ_LENGTH, max_new_words=14,
                         beam_width=width, temperature=temp, k=20,
                         penalty=1.0, device=device) for _ in range(2)]
        print(f'=== 빔 너비 {width} / 온도 {temp} / Top-20 / 페널티 1.0 ===')
        for o in outs:
            print('  ' + o)
        print()

빔 너비와 온도 조합 실험(프롬프트: 'dorothy looked at' )



=== 빔 너비 3 / 온도 0.7 / Top-20 / 페널티 1.0 ===
  dorothy looked at her companions , and put them back to the road again rushed forward ,
  dorothy looked at her companions , and put them back to the little gray mass and toto



=== 빔 너비 3 / 온도 1.0 / Top-20 / 페널티 1.0 ===
  dorothy looked at her companions and so that she saw neither the scarecrow , she said to
  dorothy looked at her companions , and put them back to the little gray mass and toto



=== 빔 너비 3 / 온도 1.3 / Top-20 / 페널티 1.0 ===
  dorothy looked at her , and toto found that he was so much . there was a
  dorothy looked at last , and they walked along he sang tol de ri de oh ,



=== 빔 너비 5 / 온도 0.7 / Top-20 / 페널티 1.0 ===
  dorothy looked at her , and they walked along he sang tol de ri de oh ,
  dorothy looked at her , and they walked along he sang tol de ri de oh ,



=== 빔 너비 5 / 온도 1.0 / Top-20 / 페널티 1.0 ===
  dorothy looked at her , and they walked along he sang tol de ri de oh ,
  dorothy looked at her , and they walked along he sang tol de ri de oh ,



=== 빔 너비 5 / 온도 1.3 / Top-20 / 페널티 1.0 ===
  dorothy looked at her , and they walked along he sang tol de ri de oh ,
  dorothy looked at her , and they walked along he sang tol de ri de oh ,



### 풀이 해설 — 연습 문제 10-13

**결합할 때 반드시 정해야 하는 것이 하나 있다. 온도를 어디에 적용할 것인가다.**

- **후보를 뽑을 때만 적용** — 각 경로에서 다음 토큰 후보를 고를 때 확률적으로 뽑되,
  경로 점수는 원래 로그 확률로 누적한다.
- 경로 점수까지 온도로 바꾸면 **경로끼리 비교가 무너진다.** 온도가 높을수록 모든
  경로의 점수가 평평해져 빔 서치가 사실상 무작위 선택이 된다.

이 풀이는 앞의 방식을 택했다. `base_log_probs`(온도 없음)로 점수를 쌓고,
`sample_logits`(온도 적용)로 후보를 뽑는다. **탐색의 다양성과 평가의 일관성을 분리**하는
것이 요점이다.

`torch.multinomial(..., replacement=False)`로 중복 없이 뽑는 것도 필요하다. 복원 추출을
쓰면 한 경로에서 같은 토큰이 여러 번 뽑혀 빔이 실질적으로 좁아진다.

**★ 그런데 조합을 바꿔도 결과가 거의 변하지 않는다.**

빔 너비 5에서는 온도 0.7, 1.0, 1.3의 결과가 **모두 동일하다.** 빔 너비 3에서만 조금씩
달라진다.

**원인은 [연습 문제 10-12]에서 확인한 대로다.** 상위 5개 토큰이 확률의 100%를 차지할
만큼 분포가 뾰족해, **온도를 올려도 뽑히는 토큰이 달라지지 않는다.** 게다가 Top-20까지
얹으면 꼬리가 아예 잘려 나간다. 빔 너비가 클수록 여러 경로가 같은 곳으로 수렴하므로
차이가 더 사라진다.

**지문이 "조합을 바꿔도 결과가 달라지지 않을 수 있다. 그렇다면 그 한계가 어디서 왔는지,
모델이 만드는 확률 분포를 직접 확인해 보자"라고 한 것이 정확히 이 지점이다.** 한계는
디코딩 방법이 아니라 **모델**에 있다.
생성 전략은 모델이 만든 확률 분포를 다시 해석할 뿐, **모델이 배우지 못한 것을 만들어
내지는 못한다.**

**결합 자체의 원리는 성립한다.** 분포에 여유가 있는 모델이라면 다음과 같은 경향이
나타난다.

| 설정 | 성격 |
|---|---|
| 낮은 온도 + 좁은 빔 | 탐욕 디코딩에 가깝다. 안정적이지만 매번 비슷하다 |
| 낮은 온도 + 넓은 빔 | 문장이 매끄럽지만 학습 데이터의 흔한 표현으로 수렴한다 |
| 높은 온도 + 좁은 빔 | 가장 변화가 크다. 대신 어색한 조합이 남을 수 있다 |
| 높은 온도 + 넓은 빔 | **온도가 후보의 다양성을 만들고, 빔 서치가 나쁜 것을 걸러 낸다** |

마지막 조합이 좋은 이유는 두 기법이 서로의 약점을 덮기 때문이다. 온도 샘플링만 쓰면
뽑은 토큰을 되돌릴 수 없지만, 빔 서치를 얹으면 여러 경로를 끝까지 끌고 가 본 뒤 고를
수 있다.

**학습 에포크를 20 안팎으로 줄인 모델**로 같은 실험을 하면 이 경향을 관찰할 수 있다.


## 연습 문제 10-14

In [11]:

# 슬라이딩 윈도우 데이터셋(7장 OzWriter 방식: 한 토큰씩 이동)
class OzSlidingDataset(Dataset):
    def __init__(self, token_idxs, seq_length):
        self.token_idxs = token_idxs
        self.seq_length = seq_length

    def __len__(self):
        return len(self.token_idxs) - self.seq_length

    def __getitem__(self, idx):
        chunk = self.token_idxs[idx:idx + self.seq_length + 1]
        return torch.tensor(chunk, dtype=torch.long)


train_set_sw = OzSlidingDataset(token_idxs[:split], SEQ_LENGTH)
valid_set_sw = OzSlidingDataset(token_idxs[split:], SEQ_LENGTH)
train_loader_sw = DataLoader(train_set_sw, batch_size=BATCH_SIZE, shuffle=True)
valid_loader_sw = DataLoader(valid_set_sw, batch_size=BATCH_SIZE, shuffle=False)

print(f'비중첩 분할   훈련 {len(train_set):>6,}개  (에포크당 {len(train_loader):>4}스텝)')
print(f'슬라이딩 윈도우 훈련 {len(train_set_sw):>6,}개  (에포크당 {len(train_loader_sw):>4}스텝)')
print(f'같은 에포크 수에서 슬라이딩 윈도우가 {len(train_set_sw) / len(train_set):.0f}배 많이 갱신한다.')

비중첩 분할   훈련  1,107개  (에포크당   18스텝)
슬라이딩 윈도우 훈련 35,407개  (에포크당  554스텝)
같은 에포크 수에서 슬라이딩 윈도우가 32배 많이 갱신한다.


In [12]:

# 같은 에포크 수(100)로 슬라이딩 윈도우 모델 학습
common.set_seed(SEED)
model_sw = build_model()
optimizer_sw = optim.Adam(model_sw.parameters(), lr=LR)

log_sw = train_loop(model_sw, train_loader_sw, valid_loader_sw,
                    nn.CrossEntropyLoss(), optimizer_sw,
                    epochs=EPOCHS, device=device, name='슬라이딩 윈도우')

슬라이딩 윈도우 학습


 에포크    훈련 손실    검증 손실     시간
  1/100       4.5850       5.0199     0:04


 10/100       1.1543       8.3807     0:47


 20/100       0.7706      10.2247     1:32


 30/100       0.6440      11.2946     2:15


...

 60/100       0.5106      13.0612     4:17


 70/100       0.4873      13.2634     5:03


 80/100       0.4716      13.6461     5:42


 90/100       0.4591      14.0779     6:21


100/100       0.4471      14.1183     7:00
------------------------------------------
최적 1 에포크 · 검증 손실 5.0199 · 전체 학습 시간 7:00


In [13]:

# 두 방식의 학습 곡선과 생성 품질 비교
print('학습 곡선 비교(에포크별 검증 손실)')
print(f"{'에포크':>6} {'비중첩':>10} {'슬라이딩':>10}")
print('-' * 30)
for ep in (1, 5, 10, 14, 20, 40, 70, 100):
    a = log_base.history['검증 손실'][ep - 1]
    b = log_sw.history['검증 손실'][ep - 1]
    print(f'{ep:>6} {a:>10.4f} {b:>10.4f}')

print()
print('최저 검증 손실')
base_best = min(log_base.history['검증 손실'])
sw_best = min(log_sw.history['검증 손실'])
print(f'  비중첩 분할   {base_best:.4f} ({log_base.history["검증 손실"].index(base_best) + 1} 에포크)')
print(f'  슬라이딩 윈도우 {sw_best:.4f} ({log_sw.history["검증 손실"].index(sw_best) + 1} 에포크)')

print()
print('생성 품질 비교(탐욕 디코딩, 20 토큰)')
for name, m in (('비중첩 분할', model), ('슬라이딩 윈도우', model_sw)):
    print(f'  [{name}]')
    for p in (PROMPT_A, PROMPT_B):
        print('    ' + generate_greedy(m, p, vocab, SEQ_LENGTH,
                                       max_new_words=20, device=device))

학습 곡선 비교(에포크별 검증 손실)
   에포크        비중첩       슬라이딩
------------------------------
     1     6.0981     5.0199
     5     5.6438     6.8584
    10     5.2171     8.3807
    14     5.1317     9.1941
    20     5.2102    10.2247
    40     5.9658    11.9268
    70     7.1292    13.2634
   100     7.9784    14.1183

최저 검증 손실
  비중첩 분할   5.1317 (14 에포크)
  슬라이딩 윈도우 5.0199 (1 에포크)

생성 품질 비교(탐욕 디코딩, 20 토큰)
  [비중첩 분할]
    dorothy looked at her companions , and put them back to the little gray mass and toto put his cold toto and held
    in the middle of the water , looking very lonely and sad . what can we do to save him ? asked dorothy
  [슬라이딩 윈도우]
    dorothy looked at her in wonder , the witch began to shrink and fall away . see what you have done ! she
    in the middle of the river . i am afraid i shall never have any brains , after all ! down the stream


### 풀이 해설 — 연습 문제 10-14

**두 방식이 만드는 샘플 수가 결정적인 차이다.** 비중첩 분할은 `(N-1) // seq_length`개,
슬라이딩 윈도우는 `N - seq_length`개를 만든다. `seq_length`가 32이므로 **샘플 수가 약
32배**이고, 같은 에포크 수라면 파라미터 갱신 횟수도 32배다.

따라서 "같은 에포크 수에서 비교"라는 지문의 조건은 **공정한 비교가 아니다.** 그리고
그것이 이 문제의 요점이다. 에포크라는 단위가 두 방식에서 서로 다른 양의 학습을 뜻한다는
것을 몸으로 겪게 하는 문제다.

**본문이 말한 트레이드오프는 다음과 같이 나타난다.**

**슬라이딩 윈도우의 중복 학습** — 하나의 토큰이 `seq_length`개의 서로 다른 샘플에
등장한다. 같은 자리에 있는 토큰을 맨 앞에서도 보고, 가운데에서도 보고, 맨 뒤에서도 본다.
데이터가 많아 보이지만 **새로운 정보가 늘어난 것은 아니다.** 그래서 검증 손실이 훨씬
빨리 내려가지만 과적합도 훨씬 빨리 온다. 한 권 분량 말뭉치에서는 이 편이 두드러진다.

**비중첩 분할의 경계 토큰 문제** — 각 샘플의 앞부분 토큰은 언제나 **짧은 문맥만** 보고
예측하도록 학습된다. 샘플 첫 토큰은 문맥이 하나도 없고, 두 번째는 하나뿐이다. 32개
토큰 중 앞쪽 몇 개가 늘 불리한 조건에 놓인다. 슬라이딩 윈도우에서는 같은 토큰이 다른
샘플에서 충분한 문맥과 함께 다시 등장하므로 이 손해가 상쇄된다.

**어느 쪽을 쓸지는 데이터 양이 정한다.** GPT 계열이 비중첩 분할을 표준으로 삼는 것은
말뭉치가 충분히 커서 중복 학습으로 얻을 것이 없고, 계산 자원을 아끼는 편이 이득이기
때문이다. 이 예제처럼 말뭉치가 작다면 슬라이딩 윈도우가 데이터를 더 짜내는 쪽이다.
다만 과적합이 빨라지므로 검증 손실을 보며 일찍 멈춰야 한다.


## 연습 문제 10-15 [도전 문제]

In [14]:

# 글자 단위 토큰화
char_text = re.sub(r'\s+', ' ', normalized).strip()
char_vocab = {c: i for i, c in enumerate(sorted(set(char_text)))}
char_itos = {i: c for c, i in char_vocab.items()}
char_idxs = [char_vocab[c] for c in char_text]

# 단어 32개에 해당하는 글자 수를 대략 맞춘다(평균 단어 길이 + 공백)
CHAR_SEQ_LENGTH = 128
CHAR_MAX_LENGTH = CHAR_SEQ_LENGTH + 1

print(f'단어 단위: 어휘 {vocab_size:>5,}개, 토큰 {len(tokens):>7,}개, seq_length {SEQ_LENGTH}')
print(f'글자 단위: 어휘 {len(char_vocab):>5,}개, 토큰 {len(char_idxs):>7,}개, seq_length {CHAR_SEQ_LENGTH}')
print(f'글자 어휘: {"".join(sorted(char_vocab))!r}')

char_split = int(len(char_idxs) * 0.8)
char_train = OzTransformerDataset(char_idxs[:char_split], CHAR_SEQ_LENGTH)
char_valid = OzTransformerDataset(char_idxs[char_split:], CHAR_SEQ_LENGTH)
char_train_loader = DataLoader(char_train, batch_size=BATCH_SIZE, shuffle=True)
char_valid_loader = DataLoader(char_valid, batch_size=BATCH_SIZE, shuffle=False)
print(f'훈련/검증 샘플 {len(char_train)} / {len(char_valid)}')

단어 단위: 어휘 2,902개, 토큰  44,299개, seq_length 32
글자 단위: 어휘    32개, 토큰 201,838개, seq_length 128
글자 어휘: ' !,.?abcdefghijklmnopqrstuvwxyz’'
훈련/검증 샘플 1261 / 315


In [15]:

# 글자 단위 모델 학습
common.set_seed(SEED)
model_char = OzWriterTransformer(
    vocab_size=len(char_vocab), d_model=D_MODEL, num_heads=NUM_HEADS,
    ff_dim=FF_DIM, num_layers=NUM_LAYERS, max_length=CHAR_MAX_LENGTH, dropout=DROPOUT,
).to(device)
optimizer_char = optim.Adam(model_char.parameters(), lr=LR)

log_char = train_loop(model_char, char_train_loader, char_valid_loader,
                      nn.CrossEntropyLoss(), optimizer_char,
                      epochs=EPOCHS, device=device, name='글자 단위')

글자 단위 학습


 에포크    훈련 손실    검증 손실     시간
  1/100       2.8229       2.5127     0:00


 10/100       2.2306       2.2319     0:02


 20/100       2.0426       2.0303     0:04


 30/100       1.8116       1.8007     0:06


...

 60/100       1.5247       1.5652     0:11


 70/100       1.4756       1.5341     0:13


 80/100       1.4405       1.5123     0:15


 90/100       1.4127       1.4978     0:16


100/100       1.3866       1.4884     0:18
------------------------------------------
최적 98 에포크 · 검증 손실 1.4873 · 전체 학습 시간 0:18


In [16]:

# 글자 단위 생성 함수와 결과
@torch.no_grad()
def generate_char(model, prompt, max_new_chars=120, temperature=1.0, device='cpu'):
    model.eval()
    ids = [char_vocab[c] for c in prompt.lower() if c in char_vocab]
    for _ in range(max_new_chars):
        src = torch.tensor(ids[-CHAR_SEQ_LENGTH:], dtype=torch.long)
        src = src.unsqueeze(0).to(device)
        logits = model(src)[0, -1, :]
        if temperature <= 0:
            nid = logits.argmax().item()
        else:
            probs = F.softmax(logits / temperature, dim=-1)
            nid = torch.multinomial(probs, num_samples=1).item()
        ids.append(nid)
    return ''.join(char_itos[i] for i in ids)


common.set_seed(SEED)
for temp in (0.0, 0.5, 1.0):
    label = '탐욕' if temp == 0.0 else f'온도 {temp}'
    print(f'=== 글자 단위 / {label} ===')
    print('  ' + generate_char(model_char, 'dorothy looked at', temperature=temp,
                               device=device))
    print()

print('=== 참고: 단어 단위 모델(탐욕, 30 토큰) ===')
print('  ' + generate_greedy(model, PROMPT_A, vocab, SEQ_LENGTH,
                             max_new_words=30, device=device))

=== 글자 단위 / 탐욕 ===
  dorothy looked at the wicked witch of the scarecrow. the scarecrow the witch the wicked witch of the witch of the witch of the witch was 

=== 글자 단위 / 온도 0.5 ===


  dorothy looked at the face with a lion the witch of the began of the woodman and the wicked witch watch of the scarecrow and her to the w

=== 글자 단위 / 온도 1.0 ===


  dorothy looked at so thanking nick havis she not because, peased begant don’t and the scarecrow. is lead the bladly came her straw again.

=== 참고: 단어 단위 모델(탐욕, 30 토큰) ===
  dorothy looked at her companions , and put them back to the little gray mass and toto put his cold toto and held toto in her arms and hid under the sun was


In [17]:

# 두 방식을 같은 잣대로 비교: 글자당 손실(bits per character)
import math

# 단어 단위 모델의 손실은 '토큰당'이므로 글자 수로 나눠 환산해야 비교할 수 있다
chars_per_token = len(char_text) / len(tokens)
word_best = min(log_base.history['검증 손실'])
char_best = min(log_char.history['검증 손실'])

print(f'단어 하나당 평균 글자 수: {chars_per_token:.2f}')
print()
print(f"{'모델':<12}{'검증 손실(토큰당)':>18}{'글자당 비트':>14}")
print('-' * 46)
print(f"{'단어 단위':<12}{word_best:>18.4f}{word_best / chars_per_token / math.log(2):>14.3f}")
print(f"{'글자 단위':<12}{char_best:>18.4f}{char_best / math.log(2):>14.3f}")

단어 하나당 평균 글자 수: 4.56

모델                  검증 손실(토큰당)        글자당 비트
----------------------------------------------
단어 단위                   5.1317         1.625
글자 단위                   1.4873         2.146


### 풀이 해설 — 연습 문제 10-15

**바꿔야 하는 것은 두 가지다.**

**하나, 어휘 사전 크기.** 단어 단위는 2,900개인데 글자 단위는 30개 안팎이다. 100분의 1로
줄어든다. 임베딩 계층과 분류기 계층의 파라미터가 그만큼 줄어, **모델 전체 파라미터가
크게 감소한다.** 희귀 단어 문제와 `<unk>` 처리도 함께 사라진다. 어떤 단어든 아는 글자의
조합으로 표현되기 때문이다.

**둘, 순차 데이터 길이.** 같은 분량의 글을 담으려면 글자 단위는 훨씬 긴 순차 데이터가
필요하다. 이 말뭉치의 단어 하나는 평균 5글자 남짓이므로, 단어 32개에 해당하는 문맥을
보려면 **글자로는 130개 안팎**이 필요하다. 이 풀이가 `CHAR_SEQ_LENGTH`를 128로 잡은
이유이며, `max_length`도 함께 키워야 위치 인코딩이 동작한다.

**여기서 셀프 어텐션의 비용이 문제가 된다.** 계산량이 길이의 제곱에 비례하므로,
길이가 4배가 되면 어텐션 계산은 16배가 된다. **어휘 사전에서 아낀 파라미터를 순차
데이터 길이에서 도로 내주는 셈**이다. 토큰화 단위를 고르는 일이 곧 이 둘 사이의 균형을
잡는 일이다.

**검증 손실을 그대로 비교하면 안 된다.** 단어 단위의 손실은 '토큰 하나당'이고 글자
단위도 '토큰 하나당'인데, 그 토큰이 가리키는 양이 다르다. 같은 잣대로 놓으려면
**글자당 비트**로 환산해야 한다. 위 셀의 마지막 표가 그 계산이다. 단어 단위 모델의
손실을 단어당 평균 글자 수로 나눈 뒤 `log 2`로 나눠 비트로 바꿨다.

**생성 품질에서 드러나는 차이가 가장 흥미롭다.**

- **글자 단위 모델은 철자를 배워야 한다.** 단어 단위 모델은 `dorothy`를 통째로 하나의
  토큰으로 다루므로 철자를 틀릴 수가 없다. 글자 단위 모델은 `d-o-r-o-t-h-y`를 순서대로
  맞혀야 하고, 학습이 부족하면 없는 단어를 만들어 낸다.
- 대신 글자 단위 모델은 **처음 보는 단어도 만들어 낼 수 있다.** 어휘 사전에 갇히지 않는
  것이 장점이자, 엉뚱한 철자를 낳는 위험이기도 하다.
- 한 권 분량의 작은 말뭉치에서는 **단어 단위가 유리하다.** 철자라는 층위를 배우는 데
  데이터를 쓰지 않아도 되기 때문이다. 말뭉치가 커질수록 글자 단위(그리고 그 절충안인
  서브워드 토큰화)의 이점이 살아난다.

**실무에서는 둘 사이의 절충인 서브워드 토큰화(BPE 등)를 쓴다.** 자주 쓰는 단어는 하나의
토큰으로, 드문 단어는 조각으로 나눠 **어휘 사전 크기와 순차 데이터 길이를 동시에**
적당한 수준으로 잡는다. 오늘날 대규모 언어 모델이 모두 이 방식을 쓴다.


## 연습 문제 10-16 [도전 문제]

**먼저 '만들 수 있는가'에 답해 보자.**

디코더만 사용하는 트랜스포머는 입력을 따로 받지 않고 **주어진 토큰의 다음 토큰을
예측**할 뿐이다. 정렬은 입력(뒤섞인 수열)과 출력(정렬된 수열)이 있는 변환 문제이므로
언뜻 맞지 않아 보인다.

그러나 **입력과 정답을 하나의 순차 데이터로 이어 붙이면** 풀 수 있다. `입력 = 정답`
형태로 만들어 두고, `=` 뒤쪽만 맞히도록 학습하면 된다. 크로스 어텐션이 하던 '정답이
입력을 참조하는 일'을 **셀프 어텐션이 대신한다.** 인과 마스크가 있어도 정답 쪽 토큰은
자기보다 앞에 있는 입력 전체를 볼 수 있기 때문이다.

이것이 오늘날 대규모 언어 모델이 번역, 요약, 질의응답을 모두 처리하는 방식이다.
`프롬프트 + 응답`을 한 줄로 이어 붙이고 응답 쪽만 학습한다.

세 가지를 정해야 한다.

1. **구분자** — 입력과 정답의 경계를 알리는 토큰이 필요하다. 여기서는 `=`를 쓴다.
2. **손실 범위** — 입력 부분까지 맞히도록 학습하면 정렬과 무관한 일에 용량을 쓴다.
   `=` 뒤쪽 위치만 손실에 넣는다.
3. **종료 표시** — 생성을 언제 멈출지 알려 줄 토큰이 필요하다. 여기서는 `#`를 쓴다.

In [18]:

# 정렬 데이터 생성(9-7, 9-9, 10-1과 같은 문제)
import random as pyrandom

SORT_N = 10
SORT_TOTAL = 4000
EOS_CHAR, PAD_CHAR, SEP = '#', '~', ' = '

rng = pyrandom.Random(SEED)
sort_pairs = []
for _ in range(SORT_TOTAL):
    nums = [rng.randint(1, 1000) for _ in range(SORT_N)]
    src_s = ', '.join(str(n) for n in nums)
    tgt_s = ', '.join(str(n) for n in sorted(nums))
    sort_pairs.append((src_s, tgt_s))

print('입력:', sort_pairs[0][0])
print('정답:', sort_pairs[0][1])
print('학습용 한 줄:', sort_pairs[0][0] + SEP + sort_pairs[0][1] + EOS_CHAR)

sort_chars = sorted(set(''.join(a + SEP + b + EOS_CHAR for a, b in sort_pairs)) | {PAD_CHAR})
sort_vocab = {c: i for i, c in enumerate(sort_chars)}
sort_itos = {i: c for c, i in sort_vocab.items()}
SORT_PAD = sort_vocab[PAD_CHAR]
SORT_MAX_LEN = max(len(a + SEP + b + EOS_CHAR) for a, b in sort_pairs)

print(f'\n어휘 사전 {len(sort_vocab)}개: {"".join(sort_chars)!r}')
print(f'한 줄 최대 길이: {SORT_MAX_LEN}')

입력: 655, 115, 26, 760, 282, 251, 229, 143, 755, 105
정답: 26, 105, 115, 143, 229, 251, 282, 655, 755, 760
학습용 한 줄: 655, 115, 26, 760, 282, 251, 229, 143, 755, 105 = 26, 105, 115, 143, 229, 251, 282, 655, 755, 760#

어휘 사전 15개: ' #,0123456789=~'
한 줄 최대 길이: 102


In [19]:

# 손실을 정답 구간에만 적용하는 데이터셋
class SortLMDataset(Dataset):
    def __init__(self, pairs, vocab, max_len):
        self.items = []
        for src_s, tgt_s in pairs:
            full = src_s + SEP + tgt_s + EOS_CHAR
            ids = [vocab[c] for c in full] + [vocab[PAD_CHAR]] * (max_len - len(full))
            answer_start = len(src_s) + len(SEP)   # 정답 첫 글자의 위치
            # 예측 위치 j는 full[j + 1]을 맞힌다 -> 정답 구간에 해당하는 j만 True
            mask = [answer_start <= j + 1 < len(full) for j in range(max_len - 1)]
            self.items.append((
                torch.tensor(ids[:-1], dtype=torch.long),
                torch.tensor(ids[1:], dtype=torch.long),
                torch.tensor(mask, dtype=torch.bool),
            ))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        return self.items[idx]


sort_train = SortLMDataset(sort_pairs[:3000], sort_vocab, SORT_MAX_LEN)
sort_valid = SortLMDataset(sort_pairs[3000:], sort_vocab, SORT_MAX_LEN)
sort_train_loader = DataLoader(sort_train, batch_size=32, shuffle=True)
sort_valid_loader = DataLoader(sort_valid, batch_size=32, shuffle=False)

src_t, tgt_t, mask_t = sort_train[0]
print(f'한 샘플 길이 {len(src_t)}, 손실에 쓰이는 위치 {mask_t.sum().item()}개')
print('손실 구간이 가리키는 글자:',
      repr(''.join(sort_itos[i] for i in tgt_t[mask_t].tolist())))

한 샘플 길이 101, 손실에 쓰이는 위치 48개
손실 구간이 가리키는 글자: '26, 105, 115, 143, 229, 251, 282, 655, 755, 760#'


In [20]:

# 정답 구간만 손실에 넣는 학습 함수
def sort_train_epoch(model, loader, optimizer, device):
    model.train()
    loss_sum, token_count = 0.0, 0
    for src, tgt, mask in loader:
        src, tgt, mask = src.to(device), tgt.to(device), mask.to(device)
        optimizer.zero_grad()
        logits = model(src)
        loss = F.cross_entropy(
            logits[mask], tgt[mask]        # 마스크로 정답 구간만 추려 손실 계산
        )
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        n = mask.sum().item()
        loss_sum += loss.item() * n
        token_count += n
    return loss_sum / token_count


@torch.no_grad()
def sort_validation(model, loader, device):
    model.eval()
    loss_sum, token_count, correct, total = 0.0, 0, 0, 0
    for src, tgt, mask in loader:
        src, tgt, mask = src.to(device), tgt.to(device), mask.to(device)
        logits = model(src)
        loss = F.cross_entropy(logits[mask], tgt[mask])
        n = mask.sum().item()
        loss_sum += loss.item() * n
        token_count += n
        preds = logits.argmax(dim=-1)
        # 정답 구간의 모든 글자가 맞아야 정답(교사 강제 조건이므로 참고용)
        ok = ((preds == tgt) | ~mask).all(dim=1)
        correct += ok.sum().item()
        total += src.size(0)
    return loss_sum / token_count, correct / total * 100.0


common.set_seed(SEED)
model_sort = OzWriterTransformer(
    vocab_size=len(sort_vocab), d_model=D_MODEL, num_heads=NUM_HEADS,
    ff_dim=FF_DIM, num_layers=NUM_LAYERS, max_length=SORT_MAX_LEN, dropout=DROPOUT,
).to(device)
optimizer_sort = optim.Adam(model_sort.parameters(), lr=LR)

SORT_EPOCHS = 60
sort_log = common.EpochLogger(SORT_EPOCHS, target_rows=10)
print('정렬(디코더만 사용) 학습')
for epoch in range(1, SORT_EPOCHS + 1):
    tr = sort_train_epoch(model_sort, sort_train_loader, optimizer_sort, device)
    va, acc = sort_validation(model_sort, sort_valid_loader, device)
    sort_log.row(epoch, tr, va, acc)
sort_log.summary()

정렬(디코더만 사용) 학습


에포크    훈련 손실    검증 손실    정확도(%)     시간
  1/60       1.6317       1.2913        0.00%     0:01


  6/60       0.9028       0.6782        0.00%     0:06


 12/60       0.4060       0.2816        0.40%     0:12


 18/60       0.3112       0.1846        8.30%     0:17


...

 36/60       0.1773       0.0906       37.20%     0:31


 42/60       0.1538       0.0680       53.30%     0:35


 48/60       0.1373       0.0617       55.10%     0:40


 54/60       0.1266       0.0609       53.00%     0:45


 60/60       0.1172       0.0487       63.30%     0:50
------------------------------------------------------
최적 59 에포크 · 검증 손실 0.0484 · 전체 학습 시간 0:50


In [21]:

# 자유 생성으로 평가(9-7, 9-9, 10-1과 같은 세 가지 지표)
@torch.no_grad()
def sort_generate(model, src_s, max_new=60, device='cpu'):
    model.eval()
    ids = [sort_vocab[c] for c in src_s + SEP]
    out = []
    for _ in range(max_new):
        window = ids[-(SORT_MAX_LEN - 1):]
        src = torch.tensor(window, dtype=torch.long).unsqueeze(0).to(device)
        nid = model(src)[0, -1, :].argmax().item()
        ch = sort_itos[nid]
        if ch in (EOS_CHAR, PAD_CHAR):
            break
        out.append(ch)
        ids.append(nid)
    return ''.join(out)


def parse_numbers(text):
    return [p.strip() for p in text.split(',') if p.strip()]


def evaluate_sort(model, pairs, device, limit=200):
    exact = char_hit = char_total = sorted_ok = 0
    for src_s, tgt_s in pairs[:limit]:
        gen = sort_generate(model, src_s, device=device)
        if gen == tgt_s:
            exact += 1
        for a, b in zip(gen, tgt_s):
            char_hit += (a == b)
        char_total += len(tgt_s)
        nums = parse_numbers(gen)
        if len(nums) >= 2 and all(n.isdigit() for n in nums):
            values = [int(n) for n in nums]
            if all(x <= y for x, y in zip(values, values[1:])):
                sorted_ok += 1
    n = min(limit, len(pairs))
    return exact / n * 100, char_hit / char_total * 100, sorted_ok / n * 100


ex, ch, so = evaluate_sort(model_sort, sort_pairs[3000:], device)
print(f'완전 일치 정확도    {ex:.1f}%')
print(f'글자 단위 일치율    {ch:.1f}%')
print(f'출력이 정렬된 비율  {so:.1f}%')
print()
print('생성 예시')
for src_s, tgt_s in sort_pairs[3000:3003]:
    print(f'  입력: {src_s}')
    print(f'  정답: {tgt_s}')
    print(f'  생성: {sort_generate(model_sort, src_s, device=device)}')
    print()

완전 일치 정확도    65.0%
글자 단위 일치율    94.0%
출력이 정렬된 비율  87.5%

생성 예시
  입력: 287, 922, 555, 181, 798, 765, 351, 873, 384, 275
  정답: 181, 275, 287, 351, 384, 555, 765, 798, 873, 922
  생성: 181, 275, 287, 351, 384, 555, 765, 798, 873, 922

  입력: 258, 187, 957, 537, 403, 472, 608, 404, 201, 144
  정답: 144, 187, 201, 258, 403, 404, 472, 537, 608, 957
  생성: 144, 187, 201, 258, 403, 472, 404, 537, 608, 957

  입력: 362, 847, 835, 627, 302, 191, 654, 253, 650, 975
  정답: 191, 253, 302, 362, 627, 650, 654, 835, 847, 975
  생성: 191, 253, 302, 362, 627, 654, 650, 835, 847, 975



### 풀이 해설 — 연습 문제 10-16

**답은 "만들 수 있다"이고, 필요한 것은 데이터 구성을 바꾸는 일뿐이다.** 모델 클래스는
`OzWriterTransformer`를 어휘 사전 크기와 `max_length`만 바꿔 그대로 쓴다. 인코더도,
크로스 어텐션도, 새로운 계층도 필요 없다.

**바꾼 것은 세 가지다.**

**하나, 입력과 정답을 한 줄로 잇는다.** `655, 115, ... = 26, 105, ...#` 형태다. 인과
마스크가 있어도 `=` 뒤의 토큰은 자기 앞의 **입력 전체를 셀프 어텐션으로 참조**할 수
있다. 9-3절에서 크로스 어텐션이 하던 일을 셀프 어텐션이 대신하는 셈이다.

**둘, 손실을 정답 구간으로 제한한다.** 마스크를 만들어 `logits[mask]`와 `tgt[mask]`로
추려 넣었다. 이 처리를 빼면 모델이 **'무작위 수열을 생성하는 법'까지 학습**하게 된다.
정렬과 무관한 일에 용량을 쓰는 데다, 입력 쪽은 애초에 맞힐 수 없는 난수라 손실이
줄지 않아 학습을 방해한다.

**셋, 종료 토큰을 둔다.** `<eos>` 역할을 하는 `#`가 없으면 생성을 언제 멈출지 알 수
없다. 본문 10-3절의 오즈 모델이 "정해진 개수를 생성한 뒤 멈춘다"고 한 것도 종료
토큰을 학습하지 않았기 때문인데, 정렬은 출력 길이가 입력마다 다르므로 종료 토큰이
반드시 필요하다.

**결과를 9장, 10-1절과 나란히 놓고 읽어야 이 문제의 값어치가 드러난다.** 같은 정렬
문제를 네 가지 구조로 풀어 본 셈이다.

| 문제 | 구조 | 정답이 입력을 보는 통로 |
|---|---|---|
| 9-7 | Seq2Seq (LSTM) | 고정된 콘텍스트 벡터 하나 |
| 9-9 | + 바다나우 어텐션 | 디코더가 인코더 출력 전체를 참조 |
| 10-1 | 트랜스포머(인코더-디코더) | 크로스 어텐션 |
| **10-17** | **디코더만** | **셀프 어텐션(한 줄로 이어 붙임)** |

**9-7이 정보 병목으로 무너진 이유가 여기서 다시 확인된다.** 통로가 넓어질수록
성능이 올라갔고, 마지막 방식은 통로라는 개념 자체를 없앴다. 입력과 정답이 같은
순차 데이터 안에 있으니 참조할 것이 따로 없다.

**대신 대가가 있다.** 입력과 정답을 한 줄에 담으므로 순차 데이터가 두 배로 길어지고,
셀프 어텐션의 계산량은 길이의 제곱에 비례한다. 인코더-디코더 구조라면 입력과 정답을
따로 처리해 이 비용을 나눌 수 있다. **디코더만 쓰는 구조가 단순한 대신 긴 입력에
불리한 이유**가 이것이며, 학습 노트가 언급한 `O(n²)` 문제와 같은 이야기다.


---

## 정리

10-3절의 여섯 문제는 두 갈래로 나뉜다.

**생성 전략(10-12 ~ 10-14)** — 학습이 끝난 모델을 그대로 두고 **디코딩 방법만** 바꾼다.
Top-k는 꼬리를 자르고, 반복 페널티는 빔 서치의 점수 편향을 상쇄하며, 결합 함수는 둘의
장점을 겹친다. 세 문제 모두 모델을 다시 학습하지 않는다는 점이 중요하다. **추론 시점의
선택만으로 결과가 크게 달라진다는 것**이 이 묶음의 교훈이다.

**구조와 데이터(10-15 ~ 10-17)** — 같은 모델로 데이터를 다르게 쓰거나(10-15), 토큰화
단위를 바꾸거나(10-16), 아예 다른 문제에 적용한다(10-17). 셋 다 모델 코드는 거의
그대로이고 **데이터 표현이 결과를 가른다.**
